<div style="background: linear-gradient(135deg, #1e3a8a 0%, #3b82f6 100%); padding: 30px; border-radius: 15px; text-align: center; color: white; box-shadow: 0 10px 20px rgba(0,0,0,0.2); margin-bottom: 20px;">
    <h1 style="margin: 0; font-size: 2.5em; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; font-weight: 800; letter-spacing: 2px; text-transform: uppercase;">
        Disaster Tweets Analysis
    </h1>
    <hr style="border: 0; height: 1px; background-image: linear-gradient(to right, rgba(255, 255, 255, 0), rgba(255, 255, 255, 0.75), rgba(255, 255, 255, 0)); margin: 15px 0;">
    <h2 style="margin: 0; font-size: 1.5em; opacity: 0.9; font-weight: 400; font-style: italic;">
        Modèles de Niveau 3 et 4
    </h2>
</div>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Notebook pour les
</h3>


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Configuration et Installation
</h3>


In [ ]:
import builtins
import pandas as pd

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    DagsHub & MLflow Init
</h3>


In [ ]:
# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.

_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


Initialized MLflow to track repo "Oscar-AS/disaster-tweets-project"

Repository Oscar-AS/disaster-tweets-project initialized!

MLflow activé avec succès sur DagsHub !


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Importation Données
</h3>


In [ ]:
# Importation de pandas pour la gestion des données tabulaires (DataFrame)
# Importation de 're' pour utiliser les expressions régulières (Regex) lors du nettoyage de texte
# Importation de la librairie emoji pour convertir les symboles visuels en texte compréhensible
# Importation de train_test_split pour diviser notre jeu de données (Train et Test)

# 1. Chargement des données
df = pd.read_csv("Base/tweets_clean.csv")

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Séparation des données
</h3>


In [ ]:

# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df, df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Implémentation des modèles (Réseaux de Neurones avec TensorFlow/Keras)
</h3>


In [15]:
# Importation de TensorFlow, la librairie Deep Learning principale de Google
# Importation spécifique de la couche de vectorisation de texte de Keras
# Importation de NumPy pour la manipulation avancée des tableaux mathématiques

# Définition de la taille maximale du vocabulaire autorisé (les 15000 mots les plus fréquents)
MAX_VOCAB_SIZE = 15000
# Définition de la taille maximale d'une phrase (tronquée si plus longue, remplie par des 0 si plus courte)
MAX_SEQUENCE_LENGTH = 128

# Instanciation de la couche de Vectorisation
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE, # Limite du vocabulaire
    output_mode='int',         # Chaque mot sera remplacé par un nombre entier
    output_sequence_length=MAX_SEQUENCE_LENGTH # Fixe la longueur de toutes les séquences à 128
)

# Apprentissage du vocabulaire : on lit le texte d'entraînement pour créer le dictionnaire mot -> entier
vectorizer.adapt(X_train.to_numpy())

# Fonction utilitaire pour préparer les données afin que TensorFlow s'entraîne plus vite
def prepare_tf_dataset(X, y, batch_size=32):
    # Création d'un dataset TensorFlow à partir de nos listes Python (X et y)
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    # Groupement des données en paquets (batches) de 32, et mise en mémoire cache dynamique (AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Retourne le dataset optimisé
    return dataset

# Création du Dataset d'entraînement accéléré
train_ds = prepare_tf_dataset(X_train, y_train)
# Création du Dataset de test accéléré
test_ds = prepare_tf_dataset(X_test, y_test)

# Importation du module MLflow dédié à TensorFlow
import mlflow.tensorflow
# Activation du suivi automatique (enregistrera la loss, les paramètres et les modèles à chaque epoch sans coder manuellement)
mlflow.tensorflow.autolog(log_models=True)


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Modèle LSTM (Long Short-Term Memory) Simple
</h3>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Description du modèle
</h3>
Le LSTM est l'architecture classique des réseaux de neurones pour le traitement de texte. Il appartient à la famille des Réseaux de Neurones Récurrents (RNN).

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Explication du fonctionnement
</h3>
Contrairement aux réseaux classiques qui traitent l'information d'un bloc, le LSTM lit le texte **mot par mot, de gauche à droite**. Il possède une "mémoire interne" (une bande transporteuse mathématique) qui lui permet de se souvenir du début de la phrase quand il arrive à la fin. Cela lui permet de comprendre le contexte.

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Pourquoi l'utiliser ici?
</h3>
C'est un modèle d'introduction parfait pour le Deep Learning sur du texte. Il saisit mieux le sens global qu'un simple modèle ML classique, mais reste rapide à entraîner.


In [16]:
# Importation des types de couches (layers) et de l'architecture séquentielle (models) de Keras
# Importation de la fonction générant un rapport complet de classification (F1, Precision, Recall, etc.)

# Démarrage manuel d'une instance (run) MLflow nommée "3.1_LSTM_Simple"
with mlflow.start_run(run_name="3.1_LSTM_Simple"):
    # Initialisation d'un réseau de neurones en couches successives (Sequential)
    model_lstm = models.Sequential([
        # Couche 1 : Vectorisation (transforme le texte brut en liste d'entiers)
        vectorizer,
        # Couche 2 : Embedding (transforme l'entier en un vecteur mathématique dense de dimension 64)
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64, mask_zero=True),
        # Couche 3 : LSTM (le coeur récurrent, avec 64 neurones)
        layers.LSTM(64),
        # Couche 4 : Couche finale (Dense) avec 1 neurone et une activation Sigmoid (renvoie une probabilité entre 0 et 1)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Compilation du modèle : définition de la fonction d'erreur (binary_crossentropy) et de l'optimiseur (adam)
    model_lstm.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Lancement de l'entraînement sur 3 itérations (epochs), en validant sur test_ds à chaque fin d'epoch
    model_lstm.fit(train_ds, validation_data=test_ds, epochs=3)
    
    # Prédiction sur le jeu de test. Si la probabilité est > 0.5, on classe 1 (Désastre), sinon 0 (astye(int) convertit False/True en 0/1)
    y_pred_lstm = (model_lstm.predict(test_ds) > 0.5).astype(int)
    
    print("\n--- Rapport LSTM Simple ---")
    # Affichage des résultats complets en confrontant les vraies valeurs (y_test) aux prédictions (y_pred_lstm)
    print(classification_report(y_test, y_pred_lstm))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_lstm, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_lstm, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_lstm, average=None)
    recall_cls = recall_score(y_test, y_pred_lstm, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_lstm))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_lstm, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


2026/05/06 11:39:12 WARNING mlflow.tensorflow: Encountered unexpected error while inferring batch size from training dataset: Sequential model 'sequential_6' has no defined input shape yet.
2026/05/06 11:39:13 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'ascii' codec can't decode byte 0xe2 in position 120: ordinal not in range(128)


Epoch 1/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.8241 - loss: 0.4544

285/285 ━━━━━━━━━━━━━━━━━━━━ 42s 136ms/step - accuracy: 0.8617 - loss: 0.3666 - val_accuracy: 0.8861 - val_loss: 0.3008
Epoch 2/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 19s 67ms/step - accuracy: 0.9373 - loss: 0.1808 - val_accuracy: 0.8905 - val_loss: 0.3141
Epoch 3/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 21s 75ms/step - accuracy: 0.9716 - loss: 0.0919 - val_accuracy: 0.8839 - val_loss: 0.3828


2026/05/06 11:40:36 WARNING mlflow.tensorflow: Failed to infer model signature: Invalid dtype: object
2026/05/06 11:40:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:40:39 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:41:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmp3ybgl_cc\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


72/72 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step

--- Rapport LSTM Simple ---
              precision    recall  f1-score   support

           0       0.90      0.97      0.93      1851
           1       0.78      0.52      0.63       423

    accuracy                           0.88      2274
   macro avg       0.84      0.74      0.78      2274
weighted avg       0.88      0.88      0.87      2274



2026/05/06 11:41:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:41:38 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:41:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmp1x0yqr7p\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run 3.1_LSTM_Simple at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2/runs/e027e2b04cc249c2afa4cbdbcc57116a
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Modèle TextCNN (Convolutional Neural Network 1D)
</h3>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Description du modèle
</h3>
Historiquement inventé pour l'analyse d'images (pour détecter des bords, des formes), le CNN a été adapté pour le texte (TextCNN) avec un succès retentissant.

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Explication du fonctionnement
</h3>
Au lieu de lire mot par mot, le CNN utilise des "fenêtres glissantes" (Filtres Convolutifs) qui regardent des groupes de 3, 4 ou 5 mots à la fois. Le modèle cherche spécifiquement des **"motifs" ou "expressions clés"** (ex: "building on fire", "heavy earthquake") indépendamment de leur position dans la phrase.


In [17]:
# Démarrage d'un nouveau Run MLflow nommé "3.2_TextCNN"
with mlflow.start_run(run_name="3.2_TextCNN"):
    # Création d'un nouveau modèle en couches empilées
    model_cnn = models.Sequential([
        # Couche 1 : Vectorisation (texte vers entiers)
        vectorizer,
        # Couche 2 : Embedding (entiers vers vecteurs mathématiques)
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64),
        # Couche 3 : Convolution 1D, qui utilise 64 filtres et lit des paquets de 5 mots (kernel_size=5) avec une fonction d'activation ReLU
        layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        # Couche 4 : Regroupement Max Global (ne garde que l'information la plus importante détectée par la convolution)
        layers.GlobalMaxPooling1D(),
        # Couche 5 : Couche dense de 32 neurones pour interpréter l'information
        layers.Dense(32, activation='relu'),
        # Couche 6 : Dropout (désactive aléatoirement 50% des neurones pour éviter le surapprentissage)
        layers.Dropout(0.5),
        # Couche 7 : Prédiction binaire finale (1 neurone, Sigmoid)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Préparation du modèle avec l'optimiseur adam et suivi de l'accuracy
    model_cnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Entraînement pendant 3 epochs
    model_cnn.fit(train_ds, validation_data=test_ds, epochs=3)
    
    # Récupération des prédictions formatées en 0 ou 1
    y_pred_cnn = (model_cnn.predict(test_ds) > 0.5).astype(int)
    
    # Affichage dans la console
    print("\n--- Rapport TextCNN ---")
    # Impression du rapport final (F1, Precision, Recall)
    print(classification_report(y_test, y_pred_cnn))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_cnn, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_cnn, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_cnn, average=None)
    recall_cls = recall_score(y_test, y_pred_cnn, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_cnn))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_cnn, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


2026/05/06 11:42:08 WARNING mlflow.tensorflow: Encountered unexpected error while inferring batch size from training dataset: Sequential model 'sequential_7' has no defined input shape yet.
2026/05/06 11:42:09 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'ascii' codec can't decode byte 0xe2 in position 120: ordinal not in range(128)


Epoch 1/3
283/285 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8133 - loss: 0.5034

285/285 ━━━━━━━━━━━━━━━━━━━━ 31s 92ms/step - accuracy: 0.8311 - loss: 0.4350 - val_accuracy: 0.8747 - val_loss: 0.3055
Epoch 2/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9148 - loss: 0.2310 - val_accuracy: 0.8843 - val_loss: 0.3178
Epoch 3/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.9642 - loss: 0.1093 - val_accuracy: 0.8839 - val_loss: 0.3862


2026/05/06 11:42:57 WARNING mlflow.tensorflow: Failed to infer model signature: Invalid dtype: object
2026/05/06 11:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:43:01 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:43:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpb4ax6h_r\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step

--- Rapport TextCNN ---
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1851
           1       0.73      0.59      0.65       423

    accuracy                           0.88      2274
   macro avg       0.82      0.77      0.79      2274
weighted avg       0.88      0.88      0.88      2274



2026/05/06 11:43:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:43:57 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:44:18 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmp77k5hqdm\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run 3.2_TextCNN at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2/runs/3cce03bdbd504585a361b8b6ad27cb1d
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Modèle BiLSTM + GRU avec Poids de Classe
</h3>

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Description du modèle
</h3>
C'est une version sous stéroïdes du LSTM, combinant plusieurs architectures récurrentes.

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    # **Explication du fonctionnement
</h3>
1. **BiDirectional :** Au lieu de lire la phrase uniquement de gauche à droite, la couche BiLSTM la lit *aussi* de droite à gauche en même temps. Cela permet de comprendre qu'un mot a été influencé par la fin de la phrase.
2. **Couche GRU :** Un parent plus moderne et plus léger du LSTM, ajouté ici pour extraire une dernière série d'informations.
3. **Class Weights :** Nous modifions l'équation d'entraînement pour que le modèle soit puni sévèrement s'il rate un tweet de "Désastre" (classe 1).



In [19]:
# Calcul mathématique pour compenser le déséquilibre des classes
# Récupération du nombre total d'exemples d'entraînement
total = len(y_train)
# Comptage du nombre d'exemples positifs (les 1, donc les vrais désastres)
pos = sum(y_train)
# Comptage du nombre d'exemples négatifs (les 0)
neg = total - pos

# Poids pour la classe 0 : un petit nombre (car la classe est majoritaire)
weight_for_0 = (1 / neg) * (total / 2.0)
# Poids pour la classe 1 : un grand nombre (car la classe est minoritaire)
weight_for_1 = (1 / pos) * (total / 2.0)
# Dictionnaire regroupant ces poids pour TensorFlow
class_weights = {0: weight_for_0, 1: weight_for_1}

# Lancement du Run MLflow pour le modèle optimisé
with mlflow.start_run(run_name="3.3_BiLSTM_Optimized"):
    # Architecture avancée du réseau
    model_opt = models.Sequential([
        # Vectorisation
        vectorizer,
        # Embedding plus puissant (128 dimensions au lieu de 64). mask_zero=True ignore le padding vide.
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=128, mask_zero=True),
        # La couche Bidirectional enveloppe le LSTM. 'return_sequences=True' permet d'enchaîner avec le GRU
        layers.Bidirectional(layers.LSTM(64, return_sequences=True)),
        # Dropout de 30% pour forcer le modèle à généraliser
        layers.Dropout(0.3),
        # Couche GRU, qui est un RNN plus rapide
        layers.GRU(32),
        # Une couche Dense classique pour extraire les conclusions
        layers.Dense(32, activation='relu'),
        # Un dernier Dropout avant la sortie
        layers.Dropout(0.3),
        # Sortie binaire
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Utilisation d'un "Learning Rate" (taux d'apprentissage) personnalisé, plus faible que celui par défaut
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
    # Compilation
    model_opt.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    # Entraînement sur 5 epochs, EN INCLUANT l'argument 'class_weight' défini plus haut
    model_opt.fit(train_ds, validation_data=test_ds, epochs=5, class_weight=class_weights)
    
    # Transformation des probabilités en entiers 0/1
    y_pred_opt = (model_opt.predict(test_ds) > 0.5).astype(int)
    
    # Affichage
    print("\n--- Rapport BiLSTM Optimisé ---")
    print(classification_report(y_test, y_pred_opt))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_opt, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_opt, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_opt, average=None)
    recall_cls = recall_score(y_test, y_pred_opt, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_opt))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_opt, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


2026/05/06 11:52:20 WARNING mlflow.tensorflow: Encountered unexpected error while inferring batch size from training dataset: Sequential model 'sequential_8' has no defined input shape yet.
2026/05/06 11:52:20 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'ascii' codec can't decode byte 0xe2 in position 120: ordinal not in range(128)


Epoch 1/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 123ms/step - accuracy: 0.6722 - loss: 0.6883

285/285 ━━━━━━━━━━━━━━━━━━━━ 119s 393ms/step - accuracy: 0.7503 - loss: 0.6674 - val_accuracy: 0.8083 - val_loss: 0.4622
Epoch 2/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 116ms/step - accuracy: 0.7953 - loss: 0.4816

285/285 ━━━━━━━━━━━━━━━━━━━━ 70s 245ms/step - accuracy: 0.8301 - loss: 0.4321 - val_accuracy: 0.8654 - val_loss: 0.3774
Epoch 3/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.9044 - loss: 0.3046

285/285 ━━━━━━━━━━━━━━━━━━━━ 73s 257ms/step - accuracy: 0.9119 - loss: 0.2747 - val_accuracy: 0.8799 - val_loss: 0.3173
Epoch 4/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 37s 128ms/step - accuracy: 0.9542 - loss: 0.1595 - val_accuracy: 0.8795 - val_loss: 0.3668
Epoch 5/5
285/285 ━━━━━━━━━━━━━━━━━━━━ 35s 124ms/step - accuracy: 0.9808 - loss: 0.0847 - val_accuracy: 0.8804 - val_loss: 0.4513


2026/05/06 11:57:56 WARNING mlflow.tensorflow: Failed to infer model signature: Invalid dtype: object
2026/05/06 11:57:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:58:01 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:58:12 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpilltexuj\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


72/72 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step

--- Rapport BiLSTM Optimisé ---
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1851
           1       0.73      0.57      0.64       423

    accuracy                           0.88      2274
   macro avg       0.82      0.76      0.78      2274
weighted avg       0.87      0.88      0.87      2274



2026/05/06 11:59:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:59:12 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:59:24 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpngy7uth_\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run 3.3_BiLSTM_Optimized at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2/runs/f1fa2aaa54b544ea98f27a859dd2a47a
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2


<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Evaluation Curves for LSTM
</h3>


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

y_probs = model_lstm.predict(test_ds).ravel()
precision, recall, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='b', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Précision-Rappel (LSTM)')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = pd.DataFrame({'y_true': y_test, 'y_prob': y_probs})
data = data.sort_values(by='y_prob', ascending=False)
data['cumulative_data_fraction'] = np.arange(1, len(data) + 1) / len(data)
data['cumulative_positive_rate'] = data['y_true'].cumsum() / data['y_true'].sum()
data['lift'] = data['cumulative_positive_rate'] / data['cumulative_data_fraction']

plt.figure(figsize=(8, 6))
plt.plot(data['cumulative_data_fraction'], data['lift'], label='Lift Curve', color='orange', lw=2)
plt.axhline(y=1, color='r', linestyle='--', label='Baseline (Random)')
plt.xlabel('Fraction of data')
plt.ylabel('Lift')
plt.title('Courbe de Lift (LSTM)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Evaluation Curves for TextCNN
</h3>


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

y_probs = model_cnn.predict(test_ds).ravel()
precision, recall, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='b', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Précision-Rappel (TextCNN)')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = pd.DataFrame({'y_true': y_test, 'y_prob': y_probs})
data = data.sort_values(by='y_prob', ascending=False)
data['cumulative_data_fraction'] = np.arange(1, len(data) + 1) / len(data)
data['cumulative_positive_rate'] = data['y_true'].cumsum() / data['y_true'].sum()
data['lift'] = data['cumulative_positive_rate'] / data['cumulative_data_fraction']

plt.figure(figsize=(8, 6))
plt.plot(data['cumulative_data_fraction'], data['lift'], label='Lift Curve', color='orange', lw=2)
plt.axhline(y=1, color='r', linestyle='--', label='Baseline (Random)')
plt.xlabel('Fraction of data')
plt.ylabel('Lift')
plt.title('Courbe de Lift (TextCNN)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

<h3 style="color: #1e40af; border-bottom: 2px solid #3b82f6; padding-bottom: 8px; margin-top: 25px; font-weight: bold; font-family: 'Segoe UI', sans-serif;">
    Evaluation Curves for BiLSTM Optimized
</h3>


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc

y_probs = model_opt.predict(test_ds).ravel()
precision, recall, _ = precision_recall_curve(y_test, y_probs)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='b', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Précision-Rappel (BiLSTM Optimized)')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data = pd.DataFrame({'y_true': y_test, 'y_prob': y_probs})
data = data.sort_values(by='y_prob', ascending=False)
data['cumulative_data_fraction'] = np.arange(1, len(data) + 1) / len(data)
data['cumulative_positive_rate'] = data['y_true'].cumsum() / data['y_true'].sum()
data['lift'] = data['cumulative_positive_rate'] / data['cumulative_data_fraction']

plt.figure(figsize=(8, 6))
plt.plot(data['cumulative_data_fraction'], data['lift'], label='Lift Curve', color='orange', lw=2)
plt.axhline(y=1, color='r', linestyle='--', label='Baseline (Random)')
plt.xlabel('Fraction of data')
plt.ylabel('Lift')
plt.title('Courbe de Lift (BiLSTM Optimized)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()